# Template: scikit-learn example vs Moxie

Copy this notebook to `notebooks/<family>/<example>.ipynb` for each verified example.

**Reference example:** `../scikit-learn/examples/<dir>/<file>.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b` (state the URL of the rendered gallery page too).

**Scenario:** what the example demonstrates, which estimators/functions/compositions it uses,
and any change made to fit Moxie (dataset bundled instead of downloaded, smaller size, typed labels, ...).

**Determinism:** say whether the reference is deterministic (fixed seeds, no threading races).
Deterministic examples need numerical parity checks; for stochastic ones compare
distribution-level quantities (scores, shapes, invariants) and explain why.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("_template/gaussian_nb_iris")

## 1. scikit-learn (Python reference)

Run the official example code, kept as close to the original as possible.

In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)
sk = GaussianNB().fit(X_train, y_train)
sk_proba = sk.predict_proba(X_test)
sk_pred = sk.predict(X_test)
print("sklearn accuracy:", sk.score(X_test, y_test))
nb.share(X_train=X_train, y_train=y_train, X_test=X_test)

sklearn accuracy: 0.9466666666666667


## 2. Moxie (native Mojo)

The same scenario through Moxie's public API. The cell is a complete Mojo program; `%%mojo` compiles it against the current assembly and runs it. Python only supplies the data files.

In [3]:
%%mojo gaussian_nb
from moxie.naive_bayes import GaussianNB
from moxie_nb_io import load, load_ints, save, save_ints


def main() raises:
    var X_train = load("X_train")
    var y_train = load_ints("y_train")
    var X_test = load("X_test")
    var model = GaussianNB()
    _ = model.fit(X_train, y_train)
    save("proba", model.predict_proba(X_test))
    save_ints("pred", model.predict(X_test))
    print("moxie classes_:", model.classes_)

moxie classes_: [0, 1, 2]


## 3. Side by side and parity

Every tolerance needs a reason. Record unsupported pieces with `nb.note(name, "GAP", detail)`.

In [4]:
mx = nb.results["gaussian_nb"]
moxie_nb.side_by_side(sklearn=sk_proba[:5], moxie=mx["proba"][:5])
nb.compare("predict", sk_pred, mx["pred"], exact=True)
nb.compare("predict_proba", sk_proba, mx["proba"], rtol=1e-9, atol=1e-12,
           reason="same Float64 algorithm; summation order differs in variance reductions")
nb.report()
nb.assert_parity()

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
predict,PASS,"(75,)","(75,)",,,exact,0 mismatches,
predict_proba,PASS,"(75, 3)","(75, 3)",8.88e-16,7.04e-15,"rtol=1e-09, atol=1e-12",,same Float64 algorithm; summation order differs in variance reductions
